# Blockwise split robustness check

Response to SYNASC 2026 Reviewer 1, who observed that control epochs are split *within* a single recording while the three noise conditions are tested on entirely separate recordings, and that with a 1000 ms epoch window at a 233 ms SOA adjacent epochs overlap (reach 4 x 233 = 932 ms). The reviewer requested "a blockwise or sub-blockwise split".

**Design** (pre-committed in `DEVIATIONS.md`, 2026-08-13, before any result was observed):

- Folds are the three control sub-blocks. Train on two, test on the third.
- Each fold's model *also* scores the full chewing / EMI / acoustic recordings, so every condition is scored by models trained on an identical quantity of data.
- Reported values are the mean across the three folds.
- Leak-free by construction: registered boundary rejection plus the inter-sub-block rest leaves an 11.9-77.8 s gap between adjacent sub-blocks, against a 932 ms overlap reach.

**Decision rule, fixed in advance.** Blockwise is promoted to primary if any of:

1. chewing-vs-control loses significance (one-tailed Wilcoxon, Holm-corrected, alpha = 0.05)
2. EMI or acoustic *gains* significance
3. the mean control ceiling **falls** by >= 2 percentage points

Otherwise the registered analysis stays primary and this is reported as a robustness line. Criterion 3 is one-directional by design: a *rise* is reported but does not promote.

Separately, per the registered section 5 contingency: a ceiling below 60% demotes classification to descriptive and promotes N170.

**Caveat carried through to the write-up.** Blockwise removes leakage *and* forces each fold to classify a target cell absent from its training set. A drop cannot be attributed to leakage alone; the blockwise ceiling is a conservative lower bound.

In [ ]:
import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
_here = Path('.').resolve()
repo_root = next((p for p in [_here, _here.parent, _here.parent.parent]
                  if (p / 'config.yaml').exists()), _here)
os.chdir(repo_root); sys.path.insert(0, str(repo_root))

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis.classifier import (
    DERIVED_PIPELINE, DERIVED_PIPELINE_BLOCKWISE,
    load_all_results, build_accuracy_dataframe, build_fold_dataframe,
    run_group_statistics, train_and_evaluate_blockwise,
)

DERIVED = 'data/derived'
print('repo root:', repo_root)

## 0. Confirm the fast SWLDA path is equivalent

`SWLDAClassifierFast` replaces the per-candidate statsmodels refit with the closed-form partitioned-regression p-value. Same mathematics, ~100x faster. This must be bit-identical or the comparison below is meaningless.

In [ ]:
from analysis._test_blockwise import run as verify_equivalence
assert verify_equivalence('01', DERIVED), 'Fast path is NOT equivalent - stop here.'

## 1. Run the blockwise CV

Resumable: subjects with an existing results JSON are skipped. Roughly 2 s per subject.

In [ ]:
from analysis.run_blockwise_cv import discover_subjects, _already_done

subjects = discover_subjects(Path(DERIVED))
pending = [s for s in subjects if not _already_done(s, Path(DERIVED))]
print(f'{len(subjects)} subjects, {len(pending)} to run')

for i, sid in enumerate(pending, 1):
    r = train_and_evaluate_blockwise(sid, derived_root=DERIVED, save=True)
    ba = r['per_condition']['control_heldout']['balanced_accuracy'] * 100
    print(f"[{i}/{len(pending)}] sub-{sid}  control {ba:5.1f}%")
print('done')

## 2. Side-by-side comparison

Both result sets share a schema, so the *same unmodified* `run_group_statistics()` runs over each. The two sets therefore differ by exactly one thing: the split.

In [ ]:
res_v3   = load_all_results(DERIVED, pipeline=DERIVED_PIPELINE)
res_bw   = load_all_results(DERIVED, pipeline=DERIVED_PIPELINE_BLOCKWISE)
print(f'v3: {len(res_v3)} subjects | blockwise: {len(res_bw)} subjects')

stats_v3 = run_group_statistics(res_v3)
stats_bw = run_group_statistics(res_bw)

CONDS = ['control_heldout', 'chewing', 'emi', 'acoustic']
rows = []
for c in CONDS:
    a, b = stats_v3['descriptives'][c], stats_bw['descriptives'][c]
    ph_a = stats_v3['wilcoxon_posthoc_vs_control'].get(c, {})
    ph_b = stats_bw['wilcoxon_posthoc_vs_control'].get(c, {})
    rows.append({
        'condition': c,
        'v3 mean': round(a['mean'], 2), 'v3 SD': round(a['sd'], 2),
        'bw mean': round(b['mean'], 2), 'bw SD': round(b['sd'], 2),
        'delta': round(b['mean'] - a['mean'], 2),
        'v3 dz': round(ph_a['cohens_d'], 2) if ph_a else None,
        'bw dz': round(ph_b['cohens_d'], 2) if ph_b else None,
        'v3 p_holm': f"{ph_a['p_one_tailed_holm']:.2g}" if ph_a else None,
        'bw p_holm': f"{ph_b['p_one_tailed_holm']:.2g}" if ph_b else None,
    })
comparison = pd.DataFrame(rows)
display(comparison)

print(f"\nFriedman  v3: chi2={stats_v3['friedman']['statistic']:.2f}, p={stats_v3['friedman']['p_value']:.3g}")
print(f"Friedman  bw: chi2={stats_bw['friedman']['statistic']:.2f}, p={stats_bw['friedman']['p_value']:.3g}")

## 3. Decision rule verdict

Evaluated mechanically against the criteria fixed in `DEVIATIONS.md` before these numbers existed.

In [ ]:
ALPHA = 0.05
CEILING_DROP_PP = 2.0
CONTINGENCY_FLOOR = 60.0

ceil_v3 = stats_v3['descriptives']['control_heldout']['mean']
ceil_bw = stats_bw['descriptives']['control_heldout']['mean']
delta   = ceil_bw - ceil_v3

ph = stats_bw['wilcoxon_posthoc_vs_control']
c1 = ph['chewing']['p_one_tailed_holm'] >= ALPHA
c2 = (ph['emi']['p_one_tailed_holm'] < ALPHA) or (ph['acoustic']['p_one_tailed_holm'] < ALPHA)
c3 = delta <= -CEILING_DROP_PP
contingency = ceil_bw < CONTINGENCY_FLOOR

print(f'control ceiling   v3 = {ceil_v3:.2f}%   blockwise = {ceil_bw:.2f}%   delta = {delta:+.2f} pp\n')
print(f'  [{"FIRED" if c1 else "  ok "}] 1. chewing lost significance      '
      f"(p_holm = {ph['chewing']['p_one_tailed_holm']:.3g})")
print(f'  [{"FIRED" if c2 else "  ok "}] 2. EMI/acoustic gained significance '
      f"(EMI {ph['emi']['p_one_tailed_holm']:.3g}, acoustic {ph['acoustic']['p_one_tailed_holm']:.3g})")
print(f'  [{"FIRED" if c3 else "  ok "}] 3. ceiling fell >= {CEILING_DROP_PP} pp        '
      f'(observed {delta:+.2f} pp)')
print(f'  [{"FIRED" if contingency else "  ok "}] S. registered 60% contingency      '
      f'(ceiling {ceil_bw:.2f}%)')

promote = c1 or c2 or c3
print('\n' + '=' * 68)
if contingency:
    print('REGISTERED CONTINGENCY TRIGGERED -> classification becomes descriptive,')
    print('N170 promoted to primary. Governed by the pre-registration, not this rule.')
elif promote:
    print('VERDICT: OPTION 2 - blockwise becomes the PRIMARY analysis.')
    print('Regenerate Table I, Table II error columns, both figures, and the')
    print('sensitivity section. Original analysis reported as the inflated original.')
else:
    print('VERDICT: OPTION 3 - registered analysis stays PRIMARY.')
    print('Blockwise reported as a robustness check alongside it.')
    if delta >= CEILING_DROP_PP:
        print(f'NOTE: ceiling ROSE {delta:+.2f} pp. Reported and flagged, but per the')
        print('one-directional rule this does not promote the blockwise analysis.')
print('=' * 68)

## 4. Does classification degrade *within* a control recording?

The operator's claim was that consumer electrode fit decays across a session, so testing inside the training recording is not the free lunch the reviewer assumes. Each fold holds out a different sub-block, so the per-fold accuracies test this directly: sub-block 0 is the freshest, sub-block 2 the most fatigued.

In [ ]:
folds = build_fold_dataframe(res_bw)
by_sb = folds.groupby('held_out_sub_block')['balanced_accuracy'].agg(['mean', 'std', 'count'])
display(by_sb.round(2))

piv = folds.pivot_table(index='subject', columns='held_out_sub_block',
                        values='balanced_accuracy').dropna()
from scipy.stats import friedmanchisquare, wilcoxon
chi2, p = friedmanchisquare(*[piv[c].values for c in piv.columns])
print(f'\nFriedman across held-out sub-blocks: chi2 = {chi2:.2f}, p = {p:.3g}  (n = {len(piv)})')
if len(piv.columns) >= 2:
    first, last = piv.columns[0], piv.columns[-1]
    _, p_fl = wilcoxon(piv[last], piv[first], alternative='less')
    print(f'sub-block {last} < sub-block {first} (one-tailed Wilcoxon): p = {p_fl:.3g}')
    print(f'  mean {first}: {piv[first].mean():.2f}%   mean {last}: {piv[last].mean():.2f}%')

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.boxplot([piv[c].values for c in piv.columns], labels=[f'sb {c}' for c in piv.columns])
ax.set_ylabel('Control balanced accuracy (%)')
ax.set_xlabel('Held-out sub-block (0 = earliest)')
ax.set_title('Within-recording degradation check')
plt.tight_layout(); plt.show()

## 5. Does control's position in the session affect the ceiling?

The reviewer's deeper point is that control is tested on the recording it trained on, while the noise conditions cross a recording boundary. The counterbalanced Latin square puts control 1st through 4th across subjects. If session position (and therefore accumulated electrode wear) does not predict the ceiling, the "same recording" advantage the concern rests on is not visible in the data.

In [ ]:
orders = pd.read_csv('protocol/order_assignments/order_assignments.csv')
ctrl_pos = (orders[orders['condition'] == 'control']
            .assign(subject=lambda d: d['subject_id'].str.replace('sub-', '', regex=False))
            [['subject', 'condition_order']])

def ceiling_frame(results, label):
    df = build_accuracy_dataframe(results)
    df = df[df['condition'] == 'control_heldout'][['subject', 'balanced_accuracy']]
    return df.rename(columns={'balanced_accuracy': label})

merged = (ceiling_frame(res_v3, 'v3')
          .merge(ceiling_frame(res_bw, 'blockwise'), on='subject')
          .merge(ctrl_pos, on='subject', how='left'))

display(merged.groupby('condition_order')[['v3', 'blockwise']]
        .agg(['mean', 'count']).round(2))

from scipy.stats import spearmanr, kruskal
for col in ['v3', 'blockwise']:
    sub = merged.dropna(subset=['condition_order'])
    rho, p_s = spearmanr(sub['condition_order'], sub[col])
    groups = [g[col].values for _, g in sub.groupby('condition_order') if len(g) > 1]
    h, p_k = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
    print(f'{col:10s} Spearman rho = {rho:+.3f} (p = {p_s:.3g}) | '
          f'Kruskal-Wallis H = {h:.2f} (p = {p_k:.3g})')
print('\nNo association => session position does not predict the ceiling.')

## 6. Numbers for the response letter

In [ ]:
out = {
    'n_subjects': stats_bw['n_subjects'],
    'ceiling_v3': round(ceil_v3, 2),
    'ceiling_blockwise': round(ceil_bw, 2),
    'ceiling_delta_pp': round(delta, 2),
    'verdict': ('contingency' if contingency else 'option_2_promote' if promote
                else 'option_3_robustness'),
    'criteria_fired': {'chewing_lost_sig': bool(c1),
                       'noise_gained_sig': bool(c2),
                       'ceiling_fell_2pp': bool(c3)},
    'blockwise': {c: {'mean': round(stats_bw['descriptives'][c]['mean'], 2),
                      'sd': round(stats_bw['descriptives'][c]['sd'], 2)} for c in CONDS},
    'v3': {c: {'mean': round(stats_v3['descriptives'][c]['mean'], 2),
               'sd': round(stats_v3['descriptives'][c]['sd'], 2)} for c in CONDS},
    'posthoc_blockwise': {c: {'dz': round(v['cohens_d'], 3),
                              'ci': [round(v['ci_low'], 3), round(v['ci_high'], 3)],
                              'p_holm': v['p_one_tailed_holm']}
                          for c, v in stats_bw['wilcoxon_posthoc_vs_control'].items()},
}
outpath = Path(DERIVED) / DERIVED_PIPELINE_BLOCKWISE / 'blockwise_comparison.json'
outpath.write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))
print('\nsaved ->', outpath)